In [3]:
import pickle

In [8]:
with open("final_model.pkl", "rb") as file:
    final_model = pickle.load(file)

print("Model loaded! ")

Model loaded! 


In [9]:
import pandas as pd
import numpy as np

# 1) Load the real inference file
inference_df = pd.read_csv("REAL_DATA.csv")
print("inference shape:", inference_df.shape)

# 2) Build the same features the model uses (date -> year/month)
inf_feat = inference_df.copy()
inf_feat["date"] = pd.to_datetime(inf_feat["date"], dayfirst=True) # DD/MM/YYYY !
inf_feat["year"] = inf_feat["date"].dt.year
inf_feat["month"] = inf_feat["date"].dt.month

# 3) Align to EXACTLY the columns the model was trained on (auto-matches, any order)
X_inf = inf_feat.reindex(columns=final_model.feature_names_in_, fill_value=0)

# 4) Predict
preds = final_model.predict(X_inf)

# 5) Business rules: closed store -> 0, no negative sales
preds = np.where(inference_df["open"].fillna(1).values == 0, 0, preds)
preds = np.clip(preds, 0, None)

# 6) Build submission: ALL original columns (incl. 'index') + new 'sales' column
submission = inference_df.copy()
submission["sales"] = preds.round().astype(int)

# 7) Save
submission.to_csv("group1.csv", index=False)

# checks
print("features fed match model:", list(X_inf.columns) == list(final_model.feature_names_in_))
print("rows kept:", submission.shape[0] == inference_df.shape[0])
print("closed-store sales all 0:", (submission.loc[inference_df['open'] == 0, 'sales'] == 0).all())
submission.head()

inference shape: (71205, 9)
features fed match model: True
rows kept: True
closed-store sales all 0: True


,index,store_ID,day_of_week,date,nb_customers_on_day,open,promotion,state_holiday,school_holiday,sales
0,272371,415,7,01/03/2015,0,0,0,0,0,0
1,558468,27,7,29/12/2013,0,0,0,0,0,0
2,76950,404,3,19/03/2014,657,1,1,0,0,6664
3,77556,683,2,29/01/2013,862,1,0,0,0,6900
4,456344,920,3,19/03/2014,591,1,1,0,0,5969
